# GMC714 - Devoir no 4 - Amorce

Cette page contient une amorce de code pour la résolution du devoir 4. Je vous recommande toutefois de travailler localement sur votre ordinateur si possible car il y a deux avantages: les graphiques en 3D peuvent être tourner pour changer la perspective et les simulations s'éxécutent beaucoup plus rapidement.

## Robot et environnement de simulation

Cette section sont des blocs de code qui définissent les propriétés du robot et de l'environnement pour la simulation.

The code bellow is simply loading some libraries:

In [ ]:
!git clone https://github.com/SherbyRobotics/pyro
import sys
sys.path.append('/content/pyro')
import pyro
from IPython import display
import numpy as np

Ici on définit le robot manipulateur utilisé pour le perçage:

In [ ]:
########################################
# Model de robot dynamiques
########################################

from pyro.dynamic.manipulator import ThreeLinkManipulator3D

class DrillingRobot( ThreeLinkManipulator3D ):
    """
    3DoF Robot manipulator
    Full dynamic model
    """

    ############################
    def __init__(self):
        """ """

        # initialize standard params
        super().__init__()

        # Name
        self.name = 'Drilling Robot'

        # kinematic
        self.l1  = 0.3
        self.l2  = 0.525
        self.l3  = 0.375

        # dynamic
        self.I1  = 0.66125
        self.m2  = 1.589
        self.m3  = 0.545

        self.gravity = 9.81


    ##############################
    def trig(self, q ):
        """
        Compute cos and sin usefull in other computation
        ------------------------------------------------
        """

        c1  = np.cos( q[0] )
        s1  = np.sin( q[0] )
        c2  = np.cos( q[1] )
        s2  = np.sin( q[1] )
        c3  = np.cos( q[2] )
        s3  = np.sin( q[2] )
        c23 = np.cos( q[2] + q[1] )
        s23 = np.sin( q[2] + q[1] )

        return [c1,s1,c2,s2,c3,s3,c23,s23]


    ##############################
    def forward_kinematic_effector(self, q ):
        """
        Task-space coord. vector
        ----------------------------------
        dim( r ) = ( dim of task-space , 1 )
        """

        [c1,s1,c2,s2,c3,s3,c23,s23] = self.trig( q )

        l1     = self.l1
        l2     = self.l2
        l3     = self.l3

        # End-effector kinematic
        x3 = c1 * ( l3 * c23 + l2 * c2)
        y3 = s1 * ( l3 * c23 + l2 * c2)
        z3 = l1 + l3 * s23 + l2 * s2

        r = np.array([x3, y3, z3])

        return r


    ##############################
    def J(self, q ):
        """
        Jacobian matrix
        ----------------------------------
        dim( J ) = ( dim of task-space , robot DoF )
        """

        [c1,s1,c2,s2,c3,s3,c23,s23] = self.trig( q )

        l2     = self.l2
        l3     = self.l3

        J = np.zeros((3,3))

        J[0,0] =  -s1*(l3*c23 + l2*c2)
        J[0,1] =  -c1*(l3*s23 + l2*s2)
        J[0,2] =  -l3*s23*c1

        J[1,0] =   c1*(l3*c23 + l2*c2)
        J[1,1] =  -s1*(l3*s23 + l2*s2)
        J[1,2] =  -l3*s23*s1

        J[2,0] =  0
        J[2,1] =  l3*c23 + l2*c2
        J[2,2] =  l3*c23

        return J


    ###########################################################################
    def H(self, q ):
        """
        Inertia matrix
        ----------------------------------
        dim( H ) = ( dof , dof )

        such that --> Kinetic Energy = 0.5 * dq^T * H(q) * dq

        """

        [c1,s1,c2,s2,c3,s3,c23,s23] = self.trig( q )

        l2     = self.l2
        l3     = self.l3

        I1      = self.I1
        m2      = self.m2
        m3      = self.m3

        H = np.zeros((3,3))

        H[0,0] = I1 + (m3*(2*(l2*c2*s1 + l3*c2*c3*s1 - l3*s1*s2*s3)**2 + 2*(l2*c1*c2 + l3*c1*c2*c3 - l3*c1*s2*s3)**2))/2 + l2**2*m2*c2**2
        H[1,0] = 0
        H[2,0] = 0

        H[0,1] = H[1,0]
        H[1,1] = (m3*(2*l2**2 + 4*c3*l2*l3 + 2*l3**2))/2 + l2**2*m2
        H[2,1] = m3*(l3**2 + l2*c3*l3)

        H[0,2] = H[2,0]
        H[1,2] = H[2,1]
        H[2,2] = l3**2*m3

        return H


    ###########################################################################
    def C(self, q , dq ):
        """
         Corriolis and Centrifugal Matrix
        ------------------------------------
        dim( C ) = ( dof , dof )

        such that --> d H / dt =  C + C^T


        """

        [c1,s1,c2,s2,c3,s3,c23,s23] = self.trig( q )

        q2 = q[1]
        q3 = q[2]

        sin=np.sin

        dq1     = dq[0]
        dq2     = dq[1]
        dq3     = dq[2]

        l2     = self.l2
        l3     = self.l3

        m2      = self.m2
        m3      = self.m3

        C = np.zeros((3,3))

        C[0,0] = 0
        C[0,1] = -dq1*(l3**2*m3*sin(2*q2 + 2*q3) + l2**2*m2*sin(2*q2) + l2**2*m3*sin(2*q2) + 2*l2*l3*m3*sin(2*q2 + q3))
        C[0,2] = -l3*m3*dq1*(l3*sin(2*q2 + 2*q3) + l2*s3 + l2*sin(2*q2 + q3))

        C[1,0] = (dq1*(l3**2*m3*sin(2*q2 + 2*q3) + l2**2*m2*sin(2*q2) + l2**2*m3*sin(2*q2) + 2*l2*l3*m3*sin(2*q2 + q3)))/2
        C[1,1] = 0
        C[1,2] = -l2*l3*m3*s3*(2*dq2 + dq3)

        C[2,0] = l3*m3*dq1*(  (l3*sin(2*q2 + 2*q3))/2 + (l2*s3)/2 + (l2*sin(2*q2 + q3)) /2   )
        C[2,1] = (l2*l3*m3*s3*(2*dq2 + dq3))/2
        C[2,2] = -(l2*l3*m3*dq2*s3)/2

        return C


    ###########################################################################
    def B(self, q ):
        """
        Actuator Matrix  : dof x m
        """

        B = np.diag( np.ones( self.dof ) ) #  identity matrix

        return B


    ###########################################################################
    def g(self, q ):
        """
        Gravitationnal forces vector : dof x 1
        """

        [c1,s1,c2,s2,c3,s3,c23,s23] = self.trig( q )

        l2 = self.l2
        l3 = self.l3

        m2 = self.m2
        m3 = self.m3

        G = np.zeros(3)

        g = self.gravity

        G[0] = 0
        G[1] = g*(m3*(l3*c23 + l2*c2) + l2*m2*c2)
        G[2] = g*l3*m3*c23

        return G


    ###########################################################################
    def d(self, q , dq ):
        """
        State-dependent dissipative forces : dof x 1
        """

        D = np.zeros((3,3))

        D[0,0] = 0.3
        D[1,1] = 0.3
        D[2,2] = 0.3

        d = np.dot( D , dq )

        return d


    ###########################################################################
    # Graphical output
    ###########################################################################

    ###########################################################################
    def forward_kinematic_domain(self, q ):
        """
        """
        l = 1.2

        domain  = [ (-l,l) , (-l,l) , (0,l*2) ]#

        return domain


    ###########################################################################
    def forward_kinematic_lines(self, q ):
        """
        Compute points p = [x;y;z] positions given config q
        ----------------------------------------------------
        - points of interest for ploting

        Outpus:
        lines_pts = [] : a list of array (n_pts x 3) for each lines

        """

        lines_pts   = [] # list of array (n_pts x 3) for each lines
        lines_style = []
        lines_color = []

        ###############################
        # ground line
        ###############################

        pts      = np.zeros(( 5 , 3 ))

        z = 0.2

        pts[0,:] = np.array([-1,-1,z])
        pts[1,:] = np.array([+1,-1,z])
        pts[2,:] = np.array([+1,+1,z])
        pts[3,:] = np.array([-1,+1,z])
        pts[4,:] = np.array([-1,-1,z])

        lines_pts.append( pts )
        lines_style.append('--')
        lines_color.append('k')

        ###########################
        # robot kinematic
        ###########################

        pts      = np.zeros(( 4 , 3 ))
        pts[0,:] = np.array([0,0,0])

        [c1,s1,c2,s2,c3,s3,c23,s23] = self.trig( q )

        # Three robot points

        l1     = self.l1
        l2     = self.l2
        l3     = self.l3

        pts[1,0] = 0
        pts[1,1] = 0
        pts[1,2] = l1

        pts[2,0] =  0 + l2 * c2 * c1
        pts[2,1] =  0 + l2 * c2 * s1
        pts[2,2] = l1 + l2 * s2

        pts[3,0] = c1 * ( l3 * c23 + l2 * c2)
        pts[3,1] = s1 * ( l3 * c23 + l2 * c2)
        pts[3,2] = l1 + l3 * s23 + l2 * s2

        lines_pts.append( pts )
        lines_style.append('o-')
        lines_color.append('b')

        return lines_pts , lines_style , lines_color


Ici on définit un environnement de perçage pour le robot:

In [ ]:
class DrillingRobotOnJig( DrillingRobot ):
    """
    Drilling robot + external force during contact & drill kinematic for graphic output

    """

    ############################
    def __init__(self):
        """ """

        super().__init__()

        self.hole_position = np.array([0.25,0.25,0.4])
        self.hole_radius   = 0.05
        self.hole_depth    = 0.2

    ##############################
    def f_ext(self, q , dq , t = 0 ):
        """
        External force due to contact during drilling

        """

        r  = self.forward_kinematic_effector( q )
        dr = self.forward_differential_kinematic_effector(q, dq)

        hole_position = self.hole_position
        hole_radius   = self.hole_radius

        # Contact:
        if r[2] < self.hole_position[2] :

            # Dans le bois
            fx = - dr[0] * 2000 # damping lateral
            fy = - dr[1] * 2000 # damping lateral
            fz = - dr[2] * 1000 # damping vertical

            # Pointe de la mèche dans le pré-trou
            if  (( r[0] > hole_position[0] - hole_radius ) &
                 ( r[0] < hole_position[0] + hole_radius ) &
                 ( r[1] > hole_position[1] - hole_radius ) &
                 ( r[1] < hole_position[1] + hole_radius ) ) :

                # Aspiration dans le trou du à l'angle de la pointe de la mèche
                ex = r[0] - hole_position[0]
                ey = r[1] - hole_position[1]
                fx = fx / 10 - 2 * ex * fz
                fy = fy / 10 - 2 * ey * fz

                # Moins de résistance verticale
                fz = fz / 2

            if r[2] < (self.hole_position[2] - self.hole_depth) :

                # Dans l'acier
                fx = - dr[0] * 10000 # damping lateral
                fy = - dr[1] * 10000 # damping lateral
                fz = - dr[2] * 10000 # damping vertical

            f_ext = np.array([fx,fy,fz])

        else:

            # No contact
            f_ext = np.zeros( self.e )

        return f_ext


    ###########################################################################
    def forward_kinematic_lines(self, q ):

        ###########################
        # Base graphic
        ###########################

        lines_pts, lines_style, lines_color = DrillingRobot.forward_kinematic_lines(self, q)

        ###########################
        # Drill
        ###########################

        [c1,s1,c2,s2,c3,s3,c23,s23] = self.trig( q )

        pts      = np.zeros(( 2 , 3 ))

        l1     = self.l1
        l2     = self.l2
        l3     = self.l3

        pts[0,0] = c1*(l3*c23 + l2*c2)
        pts[0,1] = s1*(l3*c23 + l2*c2)
        pts[0,2] = l1 + l3*s23 + l2*s2

        pts[1,0] = c1*(l3*c23 + l2*c2)
        pts[1,1] = s1*(l3*c23 + l2*c2)
        pts[1,2] = l1 + l3*s23 + l2*s2 - 0.2

        lines_pts.append( pts )
        lines_style.append('-')
        lines_color.append('r')

        ###########################
        # Hole
        ###########################

        pts      = np.zeros(( 2 , 3 ))

        x = self.hole_position[0]
        y = self.hole_position[1]
        z = self.hole_position[2]

        pts[0,:] = np.array([x,y,z-0.2])
        pts[1,:] = np.array([x,y,z-0.2 - self.hole_depth])

        lines_pts.append( pts )
        lines_style.append('--')
        lines_color.append('k')

        return lines_pts , lines_style , lines_color


## Loi de commande du robot

Dans le bloc de code ci-dessous, vous avez une amorce de code pour définir votre loi de commande qui sera testée dans la simulation.

In [ ]:
from pyro.control.robotcontrollers import EndEffectorPD

class CustomDrillingController( EndEffectorPD ) :
    """

    """

    ############################
    def __init__(self, robot_model ):
        """ """

        EndEffectorPD.__init__( self , robot_model )

        self.robot_model = robot_model

        # Label
        self.name = 'Custom Drilling Controller'

        ###################################################
        # Vos paramètres de loi de commande ici !!
        ###################################################

        # Target effector force
        self.rbar = np.array([0,0,0])



    #############################
    def c( self , y , r , t = 0 ):
        """
        Feedback static computation u = c(y,r,t)

        INPUTS
        y  : sensor signal vector     p x 1
        r  : reference signal vector  k x 1
        t  : time                     1 x 1

        OUPUTS
        u  : control inputs vector    m x 1

        """

        # Ref
        f_desired = r

        # Feedback from sensors
        x = y
        [ q , dq ] = self.x2q( x )

        # Calculs basés sur le modèle
        r = self.robot_model.forward_kinematic_effector( q ) # End-effector actual position
        J = self.robot_model.J( q )      # Jacobian
        g = self.robot_model.g( q )      # Gravity vector
        H = self.robot_model.H( q )      # Inertia matrix
        C = self.robot_model.C( q , dq ) # Coriolis matrix

        ##################################
        # Votre loi de commande ici !!!
        ##################################

        tau = np.zeros(self.m)  # place-holder de bonne dimension

        return tau

## Initialisation de l'environnement de simulation

Ici on lance une instance de notre système à contrôller:

In [ ]:
# Model dynamique du robot
sys = DrillingRobotOnJig()

# Controller
model = DrillingRobot()
ctl   = CustomDrillingController( model ) # Empty do nothing controller template

# Closed-loop dynamic
clsys = ctl + sys

Ici on définit la configuration initiale du robot:

In [ ]:
q0 = np.array([0,1.4,-1.3])    # Vecteur de la position des joints initial
clsys.x0[0:3] = q0
clsys.show3( q0 )

## Simulation d'une trajectoire

Note: ici j'ai ajuster pour avoir une discrétisation plus fine que par défault car la situation de contact avec la pièce est une dynamique assez rapide. Il est aussi à mentionner que le modèle dynamique ici de modélise pas d'impulsion lors du contact.

In [ ]:
tf = 5           # Temps final
n  = 50001       # discrétisation pour la résolution des équations différentielles
clsys.compute_trajectory( tf=tf , n=n ,  solver = 'euler' )

In [ ]:
clsys.plot_trajectory('x')

In [ ]:
clsys.plot_trajectory('u')

In [ ]:
clsys.plot_end_effector_trajectory()

In [ ]:
# Exemple extraction des données pour analyse
t        = clsys.traj.t
q_traj   = clsys.traj.x[:,0:3]  # Trajectoire des angles du robot
dq_traj  = clsys.traj.x[:,3:6]  # Trajectoire des vitesses du robot
tau_traj = clsys.traj.u         # Trajectoire des couples du robot

f_traj   = np.zeros((n,3))
for i in range(n):
    f_traj[i,:] = sys.f_ext( q_traj[i,:] , dq_traj[i,:] )

import matplotlib
fig , plots = matplotlib.pyplot.subplots(3, figsize=(6,3), dpi=200)
plots[0].plot( t , f_traj[:,0] )
plots[0].set_ylim([-1000,1000])
plots[0].grid(True)
plots[1].plot( t , f_traj[:,1] )
plots[1].set_ylim([-1000,1000])
plots[1].grid(True)
plots[2].plot( t , f_traj[:,2] )
plots[2].set_ylim([-1000,1200])
plots[2].grid(True)

## Génération d'une animation de la trajectoire

In [ ]:
# This generate an animation of the trajectory
ani  = clsys.generate_simulation_html_video( is_3d = True)
html = display.HTML( ani )
display.display(html)